# IMDB Sentiment Classification & Summarization

Comparing a custom-trained TF-IDF classifier against a pre-trained HuggingFace model, plus text summarization using BART.

**Dataset:** [IMDB Dataset of 50K Movie Reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews) (Kaggle)


## 1. Setup & Imports

In [ ]:
!pip install "transformers==4.46.3" "tokenizers>=0.20,<0.21" kagglehub -q


In [ ]:
import os
import time
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

from transformers import pipeline

# Suppress noisy download progress bars / symlink warnings on Windows
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"


## 2. Load & Clean Data

Download the CSV from the Kaggle link above and update `DATA_PATH` below to point to your local copy.

In [ ]:
import kagglehub
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
DATA_PATH = f"{path}/IMDB Dataset.csv"

In [ ]:
df = pd.read_csv(DATA_PATH)
print(df.shape)
print(df.head())
print(df['sentiment'].value_counts())


In [ ]:
df['review_clean'] = df['review'].str.lower()
df['review_clean'] = df['review_clean'].str.replace('<br />', ' ', regex=False)
df['review_clean'] = df['review_clean'].str.replace(r'[^a-z\s]', '', regex=True)
df = df.dropna()

df[['review', 'review_clean', 'sentiment']].head()


## 3. Train-Test Split & TF-IDF Vectorization

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['review_clean'], df['sentiment'], test_size=0.2, random_state=42
)

MAX_FEATURES = 5000
vectorizer = TfidfVectorizer(max_features=MAX_FEATURES)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)


## 4. Baseline Model: TF-IDF + Naive Bayes

In [ ]:
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)

nb_preds = nb_model.predict(X_test_tfidf)
nb_accuracy = accuracy_score(y_test, nb_preds)
nb_f1 = f1_score(y_test, nb_preds, pos_label='positive')

print("Naive Bayes Accuracy:", nb_accuracy)
print("Naive Bayes F1:", nb_f1)


## 5. TF-IDF + Logistic Regression

In [ ]:
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_tfidf, y_train)

lr_preds = lr_model.predict(X_test_tfidf)
lr_accuracy = accuracy_score(y_test, lr_preds)
lr_f1 = f1_score(y_test, lr_preds, pos_label='positive')

print("Logistic Regression Accuracy:", lr_accuracy)
print("Logistic Regression F1:", lr_f1)


## 6. Model Comparison: Naive Bayes vs Logistic Regression

In [ ]:
print(f"{'Model':<25}{'Accuracy':<12}{'F1 Score'}")
print(f"{'Naive Bayes':<25}{nb_accuracy:<12.4f}{nb_f1:.4f}")
print(f"{'Logistic Regression':<25}{lr_accuracy:<12.4f}{lr_f1:.4f}")


## 7. HuggingFace Pre-trained Sentiment Pipeline

Compare the custom-trained Logistic Regression model against a general-purpose pre-trained transformer.

In [ ]:
hf_classifier = pipeline("sentiment-analysis")
print("HuggingFace sentiment pipeline loaded.")


In [ ]:
# Quick side-by-side check on 5 samples
sample_reviews = X_test.iloc[:5].tolist()
sample_true_labels = y_test.iloc[:5].tolist()

for i, review in enumerate(sample_reviews):
    hf_result = hf_classifier(review, truncation=True, max_length=512)[0]
    lr_pred = lr_model.predict(vectorizer.transform([review]))[0]

    print(f"--- Review {i+1} ---")
    print(f"True label: {sample_true_labels[i]}")
    print(f"Your LR model: {lr_pred}")
    print(f"HuggingFace: {hf_result['label']} (score: {hf_result['score']:.3f})")
    print()


In [ ]:
# Larger 200-sample comparison: accuracy AND inference speed
sample_size = 200
X_sample = X_test.iloc[:sample_size].tolist()
y_sample = y_test.iloc[:sample_size].tolist()

lr_sample_preds = lr_model.predict(vectorizer.transform(X_sample))

hf_sample_preds = []
start = time.time()
for review in X_sample:
    result = hf_classifier(review, truncation=True, max_length=512)[0]
    hf_sample_preds.append(result['label'].lower())
hf_inference_time = time.time() - start

lr_sample_accuracy = accuracy_score(y_sample, lr_sample_preds)
hf_sample_accuracy = accuracy_score(y_sample, hf_sample_preds)

print(f"HuggingFace inference time: {hf_inference_time:.1f}s for {sample_size} samples")
print(f"\nYour LR model  -> Accuracy: {lr_sample_accuracy:.4f}")
print(f"HuggingFace    -> Accuracy: {hf_sample_accuracy:.4f}")


## 8. HuggingFace Summarization Pipeline (BART)

In [ ]:
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

long_review = df['review'].iloc[0]
default_summary = summarizer(long_review, max_length=100, min_length=30, do_sample=False)
print(default_summary[0]['summary_text'])


In [ ]:
short_summary = summarizer(long_review, max_length=50, min_length=15, do_sample=False)
long_summary = summarizer(long_review, max_length=150, min_length=60, do_sample=False)

print("Short summary:")
print(short_summary[0]['summary_text'])
print("\nLong summary:")
print(long_summary[0]['summary_text'])


## 9. Key Findings

- A TF-IDF + Logistic Regression model trained directly on the IMDB dataset (91.0% accuracy on a 200-sample test) outperformed a general-purpose pre-trained HuggingFace sentiment model (84.5% accuracy) on the same data, while also being dramatically faster at inference. A domain-specific model trained on in-distribution data can beat a larger pre-trained model on both accuracy and speed.
- Naive Bayes (85.1% accuracy) was outperformed by Logistic Regression (89.5% accuracy) as the classifier on top of the same TF-IDF features.
- BART summarization quality depends heavily on `max_length`: short limits can truncate output mid-sentence, since the cutoff is token-based rather than sentence-based, not a full-sentence boundary.


## 10. MLflow Experiment Tracking

Logs the already-trained Logistic Regression model (Section 5) — its hyperparameters, metrics, and the model artifact itself — instead of retraining it a second time. Pinned to MLflow 2.19.0 deliberately: newer MLflow releases (3.5.0+) introduced a Host-header security middleware, and 3.16.0's redesigned UI had a bug where the Runs list threw `INTERNAL_ERROR`. Starting on 2.19.0 from the outset avoids both issues rather than installing latest and downgrading after hitting them.

In [ ]:
!pip uninstall -y -q pyarrow
!pip install -q "mlflow==2.19.0" pyngrok pyarrow


In [ ]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("imdb-sentiment-classifier")

with mlflow.start_run(run_name="tfidf_logreg_baseline"):
    # Log the configuration actually used in Section 5 — reusing MAX_FEATURES and the
    # already-fitted lr_model rather than retraining, so this run reflects the real model.
    mlflow.log_param("max_features", MAX_FEATURES)
    mlflow.log_param("C", lr_model.C)
    mlflow.log_param("vectorizer", "TF-IDF")
    mlflow.log_param("model_type", "LogisticRegression")

    mlflow.log_metric("accuracy", lr_accuracy)
    mlflow.log_metric("f1_score", lr_f1)

    mlflow.sklearn.log_model(lr_model, "model")

    print(f"Run complete. Accuracy: {lr_accuracy:.4f}, F1: {lr_f1:.4f}")


### Start the MLflow Tracking Server + ngrok Tunnel

Uses SQLite as the backend store (the older file-based store is deprecated). No `--allowed-hosts` flag is needed here — that security middleware only exists in MLflow 3.5.0+, and we're deliberately on 2.19.0.

In [ ]:
!pkill -f mlflow
!fuser -k 5000/tcp
import time
time.sleep(3)

import subprocess

log_file = open('/content/mlflow_server.log', 'w')

mlflow_process = subprocess.Popen(
    ['mlflow', 'server',
     '--backend-store-uri', 'sqlite:///mlflow.db',
     '--host', '0.0.0.0',
     '--port', '5000'],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(8)
print("Process still running:", mlflow_process.poll() is None)


In [ ]:
from pyngrok import ngrok
from google.colab import userdata

ngrok.kill()
NGROK_TOKEN = userdata.get('NGROK_AUTHTOKEN')
ngrok.set_auth_token(NGROK_TOKEN)

public_url = ngrok.connect(5000)
print(f"MLflow UI live at: {public_url}")


### Optional — Verify Tracked Data Directly (bypasses the web UI)

Useful if the web UI ever misbehaves again: confirms the run's parameters and metrics were actually persisted, independent of any UI rendering issue.

In [ ]:
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
runs_df = mlflow.search_runs(experiment_names=["imdb-sentiment-classifier"])
print(runs_df)


In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("imdb-sentiment-classifier")

# --- Hyperparameter combinations to compare ---
experiments_to_run = [
    {"max_features": 3000, "C": 0.5},
    {"max_features": 5000, "C": 1.0},
    {"max_features": 5000, "C": 5.0},
    {"max_features": 8000, "C": 1.0},
]

for config in experiments_to_run:
    run_name = f"tfidf_logreg_mf{config['max_features']}_C{config['C']}"

    with mlflow.start_run(run_name=run_name):
        mlflow.log_param("max_features", config["max_features"])
        mlflow.log_param("C", config["C"])
        mlflow.log_param("vectorizer", "TF-IDF")
        mlflow.log_param("model_type", "LogisticRegression")

        vec = TfidfVectorizer(max_features=config["max_features"])
        X_train_vec = vec.fit_transform(X_train)
        X_test_vec = vec.transform(X_test)

        model = LogisticRegression(C=config["C"], max_iter=1000)
        model.fit(X_train_vec, y_train)

        preds = model.predict(X_test_vec)
        accuracy = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds, pos_label='positive')

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("f1_score", f1)
        mlflow.sklearn.log_model(model, "model")

        print(f"{run_name} -> Accuracy: {accuracy:.4f}, F1: {f1:.4f}")